# Pipeline — Regresión Logística
### Módulo 3 · Tema 3.1

---
## ¿Qué hace la Regresión Logística?
A pesar del nombre, **clasifica** — no predice un número.
Calcula la **probabilidad** de pertenecer a cada clase.

```
h = w₀ + w₁·x₁ + w₂·x₂ + ... + wₙ·xₙ   ← suma ponderada (igual que lineal)
P(y=1) = sigmoide(h) = 1 / (1 + e⁻ʰ)    ← convierte h a probabilidad (0 a 1)

Si P >= 0.5 → predice clase 1
Si P <  0.5 → predice clase 0
```

## ¿Cuándo usarla?
- Diagnóstico médico (maligno/benigno)
- Detección de spam
- Predicción de churn (¿el cliente se va?)
- Cualquier clasificación binaria (2 clases) o multiclase

## Métricas de evaluación
```
Accuracy  = (TP+TN) / total               ← % de aciertos
Precision = TP / (TP+FP)                  ← de los que predije positivo, ¿cuántos lo eran?
Recall    = TP / (TP+FN)                  ← de los positivos reales, ¿cuántos encontré?
F1        = 2*(Precision*Recall)/(P+R)    ← balance entre precision y recall
```

**En medicina:** el recall es crítico — no puedes dejar de detectar un cáncer real.

In [1]:
# ═══════════════════════════════════════════════════════════════
# CELDA 1 — Imports
# ═══════════════════════════════════════════════════════════════
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model    import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import StandardScaler
from sklearn.metrics         import (
    accuracy_score, confusion_matrix,
    classification_report, roc_curve, auc
)

RUTA = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'Machote'))
if RUTA not in sys.path:
    sys.path.append(RUTA)

print("Imports listos ✅")

Imports listos ✅


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 2 — Cargar datos
# ═══════════════════════════════════════════════════════════════
from sklearn.datasets import load_breast_cancer # ← CAMBIA ESTE BLOQUE por tu dataset

dataset = load_breast_cancer()
X = pd.DataFrame(dataset.data, columns=dataset.feature_names)
Y = dataset.target                   # 0=maligno, 1=benigno
CLASES = list(dataset.target_names)  # ['malignant', 'benign']

print(f"Muestras: {X.shape[0]}  |  Características: {X.shape[1]}")
print(f"Clases: {CLASES}")
vals, cnts = np.unique(Y, return_counts=True)
for v, c in zip(vals, cnts):
    print(f"  Clase {v} ({CLASES[v]}): {c} muestras ({c/len(Y)*100:.1f}%)")

Muestras: 569  |  Características: 30
Clases: [np.str_('malignant'), np.str_('benign')]
  Clase 0 (malignant): 212 muestras (37.3%)
  Clase 1 (benign): 357 muestras (62.7%)


In [3]:
# ═══════════════════════════════════════════════════════════════
# CELDA 3 — Seleccionar features útiles
# ═══════════════════════════════════════════════════════════════
df_temp = X.copy()
df_temp['__Y__'] = Y
corrs = df_temp.corr()['__Y__'].drop('__Y__').sort_values(key=abs, ascending=False)

UMBRAL = 0.5  # ← ajusta si necesitas más o menos columnas
features_utiles = corrs[corrs.abs() >= UMBRAL].index.tolist()
X_red = X[features_utiles]

print(f"Features seleccionadas ({len(features_utiles)} de {X.shape[1]}):")
for f in features_utiles:
    print(f"  {f:<35} corr={corrs[f]:+.4f}")

Features seleccionadas (15 de 30):
  worst concave points                corr=-0.7936
  worst perimeter                     corr=-0.7829
  mean concave points                 corr=-0.7766
  worst radius                        corr=-0.7765
  mean perimeter                      corr=-0.7426
  worst area                          corr=-0.7338
  mean radius                         corr=-0.7300
  mean area                           corr=-0.7090
  mean concavity                      corr=-0.6964
  worst concavity                     corr=-0.6596
  mean compactness                    corr=-0.5965
  worst compactness                   corr=-0.5910
  radius error                        corr=-0.5671
  perimeter error                     corr=-0.5561
  area error                          corr=-0.5482


In [4]:
# ═══════════════════════════════════════════════════════════════
# CELDA 4 — Dividir y escalar
# ═══════════════════════════════════════════════════════════════
# stratify=Y mantiene la misma proporción de clases en train y test
X_train, X_test, y_train, y_test = train_test_split(
    X_red, Y, test_size=0.25, stratify=Y, random_state=42
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Train: {len(X_train)}  |  Test: {len(X_test)}")

Train: 426  |  Test: 143


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 5 — Entrenar
# ═══════════════════════════════════════════════════════════════
modelo = LogisticRegression(
    max_iter=1000,   # cuántas iteraciones máximas para converger
    random_state=42
)
modelo.fit(X_train_sc, y_train)
print("Modelo entrenado ✅")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 6 — Predecir y evaluar
# ═══════════════════════════════════════════════════════════════
Y_pred  = modelo.predict(X_test_sc)
Y_proba = modelo.predict_proba(X_test_sc)  # probabilidades de cada clase

acc = accuracy_score(y_test, Y_pred)
print(f"Accuracy: {acc:.4f} ({acc*100:.1f}%)\n")
print("Reporte completo:")
print(classification_report(y_test, Y_pred, target_names=CLASES))

# Ver probabilidades de los primeros 5
print("Probabilidades (primeras 5 predicciones):")
for i in range(5):
    pred = CLASES[Y_pred[i]]
    real = CLASES[y_test[i]]
    p0   = Y_proba[i][0]
    p1   = Y_proba[i][1]
    ok   = '✅' if Y_pred[i] == y_test[i] else '❌'
    print(f"  {ok} Pred={pred:<10} Real={real:<10} P({CLASES[0]})={p0:.2f}  P({CLASES[1]})={p1:.2f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 7 — Visualizaciones
# ═══════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Gráfica 1: Matriz de confusión
cm = confusion_matrix(y_test, Y_pred)
TN, FP, FN, TP = cm.ravel()
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=[f'Pred {c}' for c in CLASES],
    yticklabels=[f'Real {c}' for c in CLASES],
    ax=axes[0]
)
axes[0].set_title(f'Matriz de Confusión — {acc*100:.1f}%')

# Gráfica 2: Curva ROC
# La curva ROC muestra el trade-off entre detectar positivos y falsos positivos
# AUC=1.0 → perfecto | AUC=0.5 → igual que adivinar al azar
fpr, tpr, _ = roc_curve(y_test, Y_proba[:, 1])
auc_score   = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='#D85A30', linewidth=2, label=f'AUC = {auc_score:.4f}')
axes[1].plot([0,1],[0,1], 'k--', linewidth=1, label='Azar (AUC=0.5)')
axes[1].set_xlabel('Tasa de Falsos Positivos')
axes[1].set_ylabel('Tasa de Verdaderos Positivos')
axes[1].set_title('Curva ROC')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"TN={TN} TP={TP} FP={FP} FN={FN}")
print(f"FP={FP} → predijo {CLASES[1]} cuando era {CLASES[0]} ← el más peligroso en medicina")